**Prompt Engineering for MediaEval 2025 NewsImage Generation Large Task**

This code uses *spacy* to extract the sentances from the text located at the URL of news articles, and *KeyBERT* further extracts the keywords from them. The output is a CSV file *"newsarticles_with_prompts.csv"*. The file is further modified in Excel to append the *"article_title"* and *"article_tags"* at the beginning because some of the URL are not found or are not accessible from our location so in that case only title and keywords were used as prompts.

Final reference file is an excel file titled *"newsarticles_with_prompts.xlsx"*, which is also supplied with the source files

In [ ]:
# Optimized for Colab with GPU support & parallel processing

!pip install -q spacy keybert trafilatura chardet
!python -m spacy download en_core_web_sm

import spacy
from keybert import KeyBERT
import trafilatura
import pandas as pd
from google.colab import drive
import chardet
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch

# Enable GPU for KeyBERT if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Mount Google Drive
drive.mount("/content/drive")

# File paths
original_csv_path = "/content/drive/MyDrive/newsarticles.csv"
output_csv_path = original_csv_path.replace(".csv", "_with_prompts.csv")

# Detect encoding
with open(original_csv_path, "rb") as f:
    result = chardet.detect(f.read())
print("Detected encoding:", result["encoding"])

# Load CSV
df = pd.read_csv(original_csv_path, encoding=result["encoding"])

# Add columns if missing
for col in ["positive_prompts", "negative_prompts"]:
    if col not in df.columns:
        df[col] = ""

# Load models
nlp = spacy.load("en_core_web_sm", disable=["ner", "tagger", "lemmatizer"])
kw_model = KeyBERT(model="distilbert-base-nli-mean-tokens")  # Smaller & faster

default_negative_prompt = (
    "blurry, distorted, text, watermark, cropped, out of frame, low resolution, "
    "duplicate, poorly drawn, deformed, noisy"
)

# Function to process a single row
def process_row(idx, url):
    try:
        downloaded = trafilatura.fetch_url(url)
        text = trafilatura.extract(downloaded)

        if not text:
            return idx, "", "NO_TEXT"

        doc = nlp(text)
        sentences = [s.text.strip() for s in doc.sents if 10 < len(s.text) < 200]

        image_prompts = []
        for sent in sentences:
            keywords = kw_model.extract_keywords(
                sent,
                keyphrase_ngram_range=(2, 4),
                stop_words="english",
                top_n=1
            )
            if keywords:
                phrase, score = keywords[0]
                if score > 0.4:
                    image_prompts.append((sent, score))

        image_prompts.sort(key=lambda x: x[1], reverse=True)
        top_prompts = [p for p, _ in image_prompts[:3]]
        pos_prompt_text = " | ".join(top_prompts)

        return idx, pos_prompt_text, "OK"

    except Exception as e:
        return idx, "", str(e)

# Parallel execution
rows_to_process = df.iloc[0:]  # Adjust as needed
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(process_row, idx, row["article_url"]) for idx, row in rows_to_process.iterrows()]
    for future in as_completed(futures):
        idx, pos_prompt, status = future.result()
        if status == "OK":
            df.at[idx, "positive_prompts"] = pos_prompt
            df.at[idx, "negative_prompts"] = default_negative_prompt
            print(f"✅ Row {idx} done")
        else:
            print(f"⚠️ Row {idx} skipped: {status}")

# Save output
df.to_csv(output_csv_path, index=False)
print(f"\n✅ New CSV saved at: {output_csv_path}")

# Unmount Drive
drive.flush_and_unmount()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 120.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Using device: cuda
Mounted at /content/drive
Detected encoding: UTF-8-SIG


ERROR:trafilatura.downloads:not a 200 response: 404 for URL https://www.rfdtv.com/helena-agri-enterprises-hosts-evolve-innovations-expo-in-memphis


⚠️ Row 0 skipped: NO_TEXT
✅ Row 1 done
✅ Row 6 done
✅ Row 4 done
✅ Row 3 done
✅ Row 7 done
✅ Row 2 done
✅ Row 10 done
✅ Row 8 done
✅ Row 11 done
✅ Row 13 done
✅ Row 14 done
✅ Row 5 done


ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.chathamdailynews.ca:443/news/local-news/former-simcoe-funeral-director-faces-criminal-charges-2


⚠️ Row 20 skipped: NO_TEXT


...

⚠️ Row 8490 skipped: NO_TEXT
✅ Row 8483 done
✅ Row 8488 done
✅ Row 8489 done
✅ Row 8494 done
✅ Row 8491 done
✅ Row 8493 done
✅ Row 8492 done
✅ Row 8496 done
✅ Row 8495 done
✅ Row 8497 done
✅ Row 8499 done
✅ Row 8487 done
✅ Row 8498 done


ERROR:trafilatura.downloads:download error: https://www.cnnphilippines.com/news/2023/7/20/hontiveros-backs-move-to-challenge-maharika.html HTTPSConnectionPool(host='www.cnnphilippines.com', port=443): Max retries exceeded with url: /news/2023/7/20/hontiveros-backs-move-to-challenge-maharika.html (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'www.cnnphilippines.com'. (_ssl.c:1016)")))


⚠️ Row 8465 skipped: NO_TEXT


ERROR:trafilatura.downloads:download error: https://www.mohavedailynews.com/news/trial-delayed-for-mohave-valley-man-accused-of-sexual-abuse-of-minor/article_6d5be9e6-24ed-11ee-9383-3ba1af29cc71.html HTTPSConnectionPool(host='www.mohavedailynews.com', port=443): Max retries exceeded with url: /news/trial-delayed-for-mohave-valley-man-accused-of-sexual-abuse-of-minor/article_6d5be9e6-24ed-11ee-9383-3ba1af29cc71.html (Caused by ResponseError('too many 429 error responses'))


⚠️ Row 8470 skipped: NO_TEXT

✅ New CSV saved at: /content/drive/MyDrive/newsarticles_with_prompts.csv
